# FlowBook: Staleness and Backward Violation

This notebook analyzes **NYC intersection traffic congestion** data.

Run cells **@A**-**@D** in order using the FlowBook Kernel.

- **@C** becomes **stale** - it trained on raw `hour` values that @D then replaced.
- **@D** shows a **backward violation** - it modified `hour`, which @C already read.

In [5]:
import pandas as pd
import numpy as np

# Generate synthetic NYC intersection traffic data
np.random.seed(42)
n_records = 50_000

# NYC grid: Avenues (1-12) run north-south, Streets (1-155) run east-west
avenues = np.random.randint(1, 13, n_records)
streets = np.random.randint(1, 156, n_records)

# Generate timestamps over 30 days with hourly readings
base_date = pd.Timestamp('2024-01-01')
timestamps = base_date + pd.to_timedelta(np.random.randint(0, 30*24, n_records), unit='h')

# Traffic directions at intersections
directions = np.random.choice(['NB', 'SB', 'EB', 'WB'], n_records)

# Congestion model: base + location effects + time effects + noise
hours = timestamps.hour
base_congestion = 30
avenue_effect = (12 - avenues) * 3  # Lower avenues (east side) more congested
street_effect = np.where(streets > 100, 5, 0)  # Uptown slightly more
rush_hour = np.where(hours.isin([7,8,9,17,18,19]), 25, 0)
noise = np.random.normal(0, 10, n_records)

congestion = np.clip(base_congestion + avenue_effect + street_effect + rush_hour + noise, 0, 100).astype(int)

df = pd.DataFrame({
    'time': timestamps,
    'avenue': avenues,
    'street': streets,
    'direction': directions,
    'congestion': congestion
})

print(f"{len(df):,} traffic readings at NYC intersections")
df.head(5)

50,000 traffic readings at NYC intersections


,time,avenue,street,direction,congestion
0,2024-01-04 00:00:00,7,124,WB,41
1,2024-01-02 06:00:00,4,138,EB,62
2,2024-01-17 06:00:00,11,70,NB,22
3,2024-01-01 08:00:00,8,138,WB,70
4,2024-01-09 06:00:00,5,106,EB,51


<div style='padding-left: 1em; font-size: 0.8em; background-color: #f0f0f8; margin-bottom: 0em;'>✓ Execute: 85 ms | Code: 43 ms | State: 37 ms | Check: 3 ms | Writes: base_congestion,avenue_effect,avenues | Stale: @B,@C</div>

In [6]:
# Extract hour feature for modeling
df['hour'] = df['time'].dt.hour

In [9]:
# Oops! Decided to convert hour to rush-hour binary flag instead
df['hour'] = df['time'].dt.hour.isin([7, 8, 9, 17, 18, 19]).astype(int)

In [10]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(df[['avenue', 'street', 'hour']], df['congestion'])
print(f"R-squared = {model.score(df[['avenue', 'street', 'hour']], df['congestion']):.3f}")

R-squared = 0.694


<div style='padding-left: 1em; font-size: 0.8em; background-color: #f0f0f8; margin-bottom: 0em;'>✓ Execute: 34 ms | Code: 16 ms | State: 12 ms | Check: 4 ms | Reads: df[avenue,congestion,hour,+1] | Writes: model,LinearRegression</div>